# AdaBoost and Random Forest refinements

This experiment asks whether a different ensemble or a less flexible forest can improve the existing PPG feature models. It keeps the original features, rows and splits so we can isolate the model change.

All new settings are chosen with three overlap-grouped folds inside training. Validation results are exploratory because we have already inspected this split in previous experiments. The old test set is not scored again.


## What changed

- **AdaBoost:** shallow trees are fitted in sequence, placing more emphasis on examples that earlier trees missed. For regression, scikit-learn combines their outputs using a weighted median. We fit separate SBP and DBP models.
- **Regularized RF:** shallower trees, larger leaves and smaller bootstrap samples reduce how closely the forest can fit individual training windows.
- **Separate-target RF:** each pressure has its own forest instead of sharing splits between SBP and DBP.
- **Baselines:** predict either the training mean or the training median for every window.

AdaBoost is worth testing, but it can spend effort on noisy labels as well as useful difficult examples. Its results need the same checks as RF. Reference: [scikit-learn AdaBoostRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostRegressor.html).


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
RUN = Path('outputs/adaboost_rf_oct28')
settings = json.loads((RUN / 'selected_settings.json').read_text())
display(pd.DataFrame(settings).T)


,params,selection_cv_mae
AdaBoost,{'regressor__model__estimator__estimator__max_...,12.068817
Regularized RF,"{'regressor__model__max_depth': 8, 'regressor_...",11.923794
Separate-target RF,{'regressor__model__estimator__max_depth': Non...,11.834446


## Validation errors

MAE is the average absolute error in mmHg. RMSE gives larger errors more weight. Lower is better. Compare every candidate with both constants and the original RF. A small difference on this repeatedly inspected validation set is not a final accuracy claim.


In [2]:
validation = pd.read_csv(RUN / 'validation_metrics.csv')
display(validation.pivot(index='model', columns='target', values=['mae_mmHg', 'rmse_mmHg']).round(2))
display(pd.read_csv(RUN / 'validation_group_bootstrap.csv').round(2))


mae_mmHg        rmse_mmHg       
target                    dbp    sbp       dbp    sbp
model                                                
AdaBoost                 9.11  17.72     12.49  22.91
Original Extra Trees     8.69  16.96     11.97  22.45
Original RF              8.83  16.86     12.16  22.44
Regularized RF           8.88  16.97     12.17  22.37
Separate-target RF       8.80  16.77     12.18  22.25
Training mean           10.03  18.70     13.26  23.43
Training median         10.21  18.93     13.92  23.83

,model,target,overlap_groups,group_macro_mae,mae_ci_low,mae_ci_high,gain_over_mean_baseline,gain_ci_low,gain_ci_high
0,AdaBoost,sbp,100,17.23,14.60,19.83,1.12,-0.07,2.40
1,Original Extra Trees,sbp,100,16.21,13.64,18.83,2.14,0.41,3.79
2,Original RF,sbp,100,16.36,13.73,18.98,1.99,0.40,3.61
3,Regularized RF,sbp,100,16.38,13.77,18.97,1.98,0.58,3.38
4,Separate-target RF,sbp,100,16.50,13.94,19.11,1.86,0.25,3.51
5,Training mean,sbp,100,18.35,15.71,21.03,0.00,0.00,0.00
6,Training median,sbp,100,18.38,15.69,21.07,-0.02,-0.52,0.48
7,AdaBoost,dbp,100,9.56,8.01,11.30,0.88,0.37,1.43
8,Original Extra Trees,dbp,100,9.08,7.52,10.81,1.37,0.69,2.07
9,Original RF,dbp,100,9.17,7.64,10.86,1.28,0.60,1.98


## Training versus validation

A much lower training error suggests the model is fitting patterns that do not transfer well to unseen recordings. Less overfitting helps only if validation errors also improve; a smaller gap alone is not success.


In [3]:
training = pd.read_csv(RUN / 'train_metrics.csv')
comparison = training[['model', 'target', 'mae_mmHg']].merge(
    validation[['model', 'target', 'mae_mmHg']], on=['model', 'target'], suffixes=('_train', '_validation'))
display(comparison.round(2))


,model,target,mae_mmHg_train,mae_mmHg_validation
0,Training mean,sbp,17.33,18.70
1,Training mean,dbp,8.38,10.03
2,Training median,sbp,17.19,18.93
3,Training median,dbp,8.21,10.21
4,Original RF,sbp,9.50,16.86
5,Original RF,dbp,4.59,8.83
6,Original Extra Trees,sbp,6.26,16.96
7,Original Extra Trees,dbp,3.15,8.69
8,AdaBoost,sbp,14.90,17.72
9,AdaBoost,dbp,7.13,9.11


## Reproduce

`compare_adaboost_rf.py` runs eight AdaBoost settings, four regularized-RF settings and four separate-target-RF settings, each on three grouped training folds. Median imputation and target scaling are fitted inside each fold. Selection minimizes the mean of SBP and DBP window MAE in mmHg; RMSE is also saved.

Run from this folder with the project environment. Use a new output directory so an earlier run is not overwritten. Original forest checkpoints are generated files; see the project README for how to recreate them. The saved CSVs can be read without model binaries.


In [4]:
RUN_NEW_EXPERIMENT = False
if RUN_NEW_EXPERIMENT:
    from compare_adaboost_rf import main
    main(Path('outputs/expanded_eda_oct14'), Path('outputs/random_forest_oct14'),
         Path('outputs/adaboost_rf_new_run'))


## Result

AdaBoost beat the mean baseline but was worse than original RF. Separate-target RF changed window MAE from 16.86/8.83 to 16.77/8.80 mmHg; equal-group errors were slightly worse. There is no clear improvement. Keep the original RF as the reference and focus the next experiment on pulse features and more distinct training recordings. See [ADABOOST_RF_RESULTS.md](ADABOOST_RF_RESULTS.md).